# Analytical Synthesis & Evidence-Gap Closure

## Purpose

This notebook consolidates the validated analytical evidence into a final research-question synthesis, identifies unresolved evidence gaps, and reconciles regulatory, platform-policy, and driver-reported deduction evidence within their documented scopes.

## Evidence Base

In [1]:

from pathlib import Path
import os
import subprocess

from google.colab import userdata

EXPECTED_HEAD = "cbd23094b18c0be5fa6ab59b4409e5123965d1ef"
REPO_URL = "https://github.com/Ronaldo-spec/indonesia-ojol-driver-economics-analysis.git"
REPO_DIR = Path("/content/indonesia-ojol-driver-economics-analysis")

assert not REPO_DIR.exists(), (
    f"{REPO_DIR} already exists. Use a fresh runtime for this notebook."
)

token = userdata.get("GITHUB_TOKEN")
assert token and token.strip(), "GITHUB_TOKEN is required in Colab Secrets for read-only repository access."

askpass_path = Path("/tmp/ojol_git_askpass.sh")
askpass_path.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\n' "x-access-token" ;;
  *Password*) printf '%s\n' "$GITHUB_TOKEN" ;;
  *) printf '\n' ;;
esac
""",
    encoding="utf-8",
)
askpass_path.chmod(0o700)

git_env = os.environ.copy()
git_env["GIT_ASKPASS"] = str(askpass_path)
git_env["GIT_TERMINAL_PROMPT"] = "0"
git_env["GITHUB_TOKEN"] = token

try:
    subprocess.run(
        ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)],
        env=git_env,
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
finally:
    askpass_path.unlink(missing_ok=True)
    git_env.pop("GITHUB_TOKEN", None)
    del token

head = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
branch = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "branch", "--show-current"],
    text=True,
).strip()

assert branch == "main", f"Unexpected branch: {branch}"
assert head == EXPECTED_HEAD, f"Remote main moved: expected {EXPECTED_HEAD}, got {head}"

print("Input repository state verified.")

import hashlib

EXPECTED_BLOBS = {
    "docs/stage0_research_design.md": "b382e5a7cb8a7d74d65e4da1f945ea64aa9a5b41",
    "metadata/stage1_source_registry.csv": "00419f6850f67201e189d7790763c431c120805b",
    "metadata/stage2_reference_input_registry.csv": "88c76678e6249a42b70da8a8df0720f5465c2c83",
    "metadata/stage3_analysis_eligibility.csv": "606b4161c75540fc831bfe679b840d71c6437311",
    "data/analytical/stage4_unit_economics_results.csv": "7c591419d2423f179e7cc687850eab8994563ea3",
    "data/analytical/stage5_findings.csv": "e0edd5d2be9162b6b93527b2c04a6f8e8223cdac",
    "metadata/stage5_visualization_registry.csv": "a652b1f1f9bf2f847f0fc010622df6c6fd697851",
    "metadata/stage5_methodological_decision_log.csv": "1b58062114f6dbb02da7b537dbd376972953aede",
    "metadata/stage5_visualization_validation.csv": "46e7e7e5f1a1fd273f0da6cbd620cf930066b62a",
    "metadata/stage5_output_manifest.csv": "89ae53666938baf1d3a2be33ee7efb7c8ccfa317",
    "metadata/stage5_closure_summary.csv": "925745059c7cdbbbec1eec3507ed997fddb031c7",
    "metadata/stage5_closure_validation.csv": "7fcf3548d0f478a44a9ebdea4367b59897a857b1",
    "visualizations/stage5_src010_income_distribution.svg": "77568f01b4d105da83fd3b1b471dcc72b6b83139",
    "visualizations/stage5_mixed_fuel_food_share.svg": "05268c682c5eff0b5e91d966afec9b4c6cc974cd",
    "visualizations/stage5_seven_day_workweek_prevalence.svg": "94af89bd1c1e92332caf05740fea9ecad9e1df75",
    "visualizations/stage5_reported_twenty_percent_deduction_prevalence.svg": "36519031bf8a200ca8d693e1ccc135432debb385",
}

def git_blob_sha(path):
    payload = path.read_bytes()
    return hashlib.sha1(f"blob {len(payload)}\0".encode() + payload).hexdigest()

for rel, expected in EXPECTED_BLOBS.items():
    path = REPO_DIR / rel
    assert path.is_file(), f"Missing committed upstream input: {rel}"
    actual = git_blob_sha(path)
    assert actual == expected, f"Blob mismatch for {rel}: expected {expected}, got {actual}"

input_integrity_verified = True
print("Input integrity verified.")


Input repository state verified.
Input integrity verified.


In [2]:
import csv
import re
import pandas as pd

A = REPO_DIR / "data" / "analytical"
M = REPO_DIR / "metadata"

def read_csv(rel):
    return pd.read_csv(
        REPO_DIR / rel,
        dtype=str,
        keep_default_na=False,
    )

def read_source_ids(rel):
    path = REPO_DIR / rel
    source_ids = []

    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        for row in reader:
            if not row:
                continue
            source_id = row[0].strip()
            if source_id == "source_id":
                continue
            if source_id:
                source_ids.append(source_id)

    result = pd.DataFrame({"source_id": source_ids})
    assert result["source_id"].ne("").all()
    assert result["source_id"].is_unique
    return result

design_text = (
    REPO_DIR / "docs" / "stage0_research_design.md"
).read_text(encoding="utf-8")

rq_section_match = re.search(
    r"## 3\. Core research questions\s*(.*?)\s*## 4\.",
    design_text,
    flags=re.S,
)
assert rq_section_match, "Research-question section was not found."

rq_matches = re.findall(
    r"^\s*(\d+)\.\s+(.+?\?)\s*$",
    rq_section_match.group(1),
    flags=re.M,
)
assert len(rq_matches) == 7, (
    f"Expected 7 research questions, found {len(rq_matches)}."
)

research_questions = {
    f"RQ{number}": question.strip()
    for number, question in rq_matches
}
assert set(research_questions) == {f"RQ{i}" for i in range(1, 8)}

s5_findings = read_csv(
    "data/analytical/stage5_findings.csv"
)
s5_registry = read_csv(
    "metadata/stage5_visualization_registry.csv"
)
s5_decisions = read_csv(
    "metadata/stage5_methodological_decision_log.csv"
)
s5_validation = read_csv(
    "metadata/stage5_visualization_validation.csv"
)
s5_manifest = read_csv(
    "metadata/stage5_output_manifest.csv"
)
s5_closure = read_csv(
    "metadata/stage5_closure_summary.csv"
)
s5_closure_validation = read_csv(
    "metadata/stage5_closure_validation.csv"
)

s4_unit = read_csv(
    "data/analytical/stage4_unit_economics_results.csv"
)
s3_eligibility = read_csv(
    "metadata/stage3_analysis_eligibility.csv"
)
s2_references = read_csv(
    "metadata/stage2_reference_input_registry.csv"
)

# Only source IDs are required from this registry for traceability.
# Reading the first field with csv.reader avoids imposing a rectangular
# pandas parse on source text fields that can contain embedded commas.
s1_sources = read_source_ids(
    "metadata/stage1_source_registry.csv"
)

assert len(s5_findings) == 7
assert len(s5_registry) == 4
assert len(s5_decisions) == 8
assert len(s5_validation) == 22
assert len(s5_manifest) == 8
assert len(s5_closure_validation) == 6

assert (
    s5_closure_validation["status"] == "PASS"
).all()

assert (
    s5_closure.iloc[0]["closure_status"]
    == "PASS_WITH_CAVEAT"
)

status_counts = (
    s5_validation["status"]
    .value_counts()
    .to_dict()
)

assert status_counts.get("PASS", 0) == 17
assert status_counts.get("CAVEAT", 0) == 5
assert status_counts.get("FAIL", 0) == 0

nonblank_comparability = set(
    s5_findings.loc[
        s5_findings["comparability_status"] != "",
        "comparability_status",
    ]
)

assert nonblank_comparability == {
    "comparable_with_caveat"
}

assert set(s4_unit["unit_economics_id"]) == {
    "S4UE001",
    "S4UE002",
}

assert set(s4_unit["derivation_basis"]) == {
    "ratio_of_source_means"
}

assert set(s4_unit["value_provenance"]) == {
    "derived"
}

assert len(s3_eligibility) == 17

eligibility_counts = (
    s3_eligibility[
        "primary_analysis_eligibility"
    ]
    .value_counts()
    .to_dict()
)

assert eligibility_counts == {
    "not_comparable": 7,
    "eligible_with_caveat": 4,
    "not_eligible_for_primary_transformed_comparison": 3,
    "context_only": 3,
}

required_reference_sources = {
    "SRC026",
    "SRC027",
    "SRC028",
}

assert required_reference_sources <= set(
    s2_references["source_id"]
)

required_source_ids = {
    "SRC003",
    "SRC004",
    "SRC005",
    "SRC006",
    "SRC007",
    "SRC010",
    "SRC013",
    "SRC026",
    "SRC027",
    "SRC028",
    "SRC029",
    "SRC030",
}

assert required_source_ids <= set(
    s1_sources["source_id"]
)

for rel in s5_manifest["output_path"]:
    assert (REPO_DIR / rel).is_file(), (
        f"Expected analytical output is missing: {rel}"
    )

print(
    "Analytical inputs validated: "
    "7 findings | 4 figures | 8 methodological decisions | "
    "22 validation checks | 17 PASS | 5 CAVEAT | 0 FAIL"
)

Analytical inputs validated: 7 findings | 4 figures | 8 methodological decisions | 22 validation checks | 17 PASS | 5 CAVEAT | 0 FAIL


## Research-Question Synthesis

In [3]:

research_question_synthesis = pd.DataFrame([
    {
        "research_question_id": "RQ1",
        "research_question": "What are driver gross earnings, deductions, operating costs, and estimated net earnings?",
        "synthesis_status": "partially_supported",
        "synthesis_answer": (
            "Validated evidence supports source-specific earnings descriptions, driver-reported deduction categories, "
            "a source-defined mixed fuel-plus-food/drink spending bundle, and two SRC029 gross unit-economics rates. "
            "Project net operating earnings are not computable because the complete same-observation gross-to-net chain is absent."
        ),
        "evidence_ids": "S5FND002;S5FND004;S5FND005;S5FND006",
        "source_ids": "SRC013;SRC029",
        "comparability_boundary": (
            "Cross-source numeric uses remain comparable_with_caveat; reported deductions are not realized transaction deductions; "
            "mixed fuel-plus-food/drink spending is not project operating cost."
        ),
        "remaining_gap_ids": "S6GAP001;S6GAP002;S6GAP003;S6GAP004",
        "intended_report_destination": "Results — earnings, deductions, cost boundary, and net reconstruction limit",
    },
    {
        "research_question_id": "RQ2",
        "research_question": "Which costs most reduce earnings?",
        "synthesis_status": "not_assessable",
        "synthesis_answer": (
            "The current evidence does not support a defensible ranking of project operating-cost components. "
            "The main cross-source spending evidence combines fuel with personal food/drink expenditure, while separate same-observation "
            "fuel and other operating-cost components are incomplete."
        ),
        "evidence_ids": "S5FND002;S5FND006",
        "source_ids": "SRC013;SRC029",
        "comparability_boundary": "The mixed source-defined bundle cannot be reclassified as project operating cost or decomposed without source support.",
        "remaining_gap_ids": "S6GAP003;S6GAP004",
        "intended_report_destination": "Results / limitations — operating-cost composition",
    },
    {
        "research_question_id": "RQ3",
        "research_question": "How do outcomes vary by workload, service, vehicle, platform, geography, and period?",
        "synthesis_status": "partially_supported",
        "synthesis_answer": (
            "Variation can be described within individual sources and through four explicitly caveated comparison uses. "
            "The validated evidence does not support a pooled national trend or defensible comparative attribution by platform, service, or vehicle."
        ),
        "evidence_ids": "S5FND001;S5FND002;S5FND003;S5FND004;S5FND007",
        "source_ids": "SRC010;SRC013;SRC029;SRC030",
        "comparability_boundary": (
            "There are zero directly_comparable cross-source uses. Recalled periods are not independent waves, and samples differ by period, "
            "geography, and recruitment."
        ),
        "remaining_gap_ids": "S6GAP007;S6GAP008;S6GAP009;S6GAP010",
        "intended_report_destination": "Results — source-bounded variation and comparability limits",
    },
    {
        "research_question_id": "RQ4",
        "research_question": "What net earnings per hour, order, and kilometer are defensible?",
        "synthesis_status": "not_computable",
        "synthesis_answer": (
            "No project net earnings rate per hour, order, or kilometer is computable. "
            "For aligned SRC029 source means only, gross service earnings equal Rp15,272.73 per source-reported working hour "
            "and Rp16,800 per completed order; both are ratios of source means. No per-kilometer earnings rate is defensible."
        ),
        "evidence_ids": "S5FND005;S5FND006;S4UE001;S4UE002",
        "source_ids": "SRC029",
        "comparability_boundary": (
            "The available rates are gross rather than net; working-hour basis remains source_reported_unspecified; "
            "distance basis is unresolved for per-kilometer derivation."
        ),
        "remaining_gap_ids": "S6GAP001;S6GAP005;S6GAP006",
        "intended_report_destination": "Results — unit economics and denominator boundaries",
    },
    {
        "research_question_id": "RQ5",
        "research_question": "What activity is required to reach specified net-income targets?",
        "synthesis_status": "not_computable",
        "synthesis_answer": (
            "Required activity for a net-income target cannot be calculated from the validated evidence because project net operating earnings "
            "and the necessary same-observation activity-to-net relationship are unavailable."
        ),
        "evidence_ids": "S5FND006",
        "source_ids": "",
        "comparability_boundary": "No target-activity scenario is substituted for missing observed or eligible model inputs.",
        "remaining_gap_ids": "S6GAP001;S6GAP005;S6GAP006",
        "intended_report_destination": "Limitations — target activity analysis",
    },
    {
        "research_question_id": "RQ6",
        "research_question": "How sensitive are results to tariffs, fuel prices, incentives, deductions, utilization, maintenance, and working hours?",
        "synthesis_status": "not_computable",
        "synthesis_answer": (
            "A defensible sensitivity model cannot be estimated from the validated evidence. "
            "Fuel cost, realized deductions, utilization-consistent time measures, maintenance, and a complete net-earnings baseline are not jointly available."
        ),
        "evidence_ids": "S5FND004;S5FND005;S5FND006",
        "source_ids": "SRC013;SRC029",
        "comparability_boundary": "Reference fuel prices or regulatory rates cannot substitute for missing driver-level realized inputs.",
        "remaining_gap_ids": "S6GAP001;S6GAP002;S6GAP003;S6GAP005",
        "intended_report_destination": "Limitations — sensitivity analysis",
    },
    {
        "research_question_id": "RQ7",
        "research_question": "What can and cannot be concluded about economic sustainability?",
        "synthesis_status": "not_assessable",
        "synthesis_answer": (
            "The evidence supports bounded statements about source-reported earnings distributions, work intensity, reported deductions, "
            "and two gross unit rates. It does not support an overall economic-sustainability conclusion because project net operating earnings "
            "are unavailable and the available samples and cross-source comparisons do not establish a nationally representative common basis."
        ),
        "evidence_ids": "S5FND001;S5FND003;S5FND004;S5FND005;S5FND006;S5FND007",
        "source_ids": "SRC010;SRC013;SRC029;SRC030",
        "comparability_boundary": "No composite welfare/sustainability score or forced overall conclusion is permitted.",
        "remaining_gap_ids": "S6GAP001;S6GAP007;S6GAP008;S6GAP009;S6GAP010",
        "intended_report_destination": "Discussion / limitations — economic sustainability boundary",
    },
])

research_question_synthesis["research_question"] = (
    research_question_synthesis["research_question_id"]
    .map(research_questions)
)
assert research_question_synthesis["research_question"].ne("").all()

regulatory_reconciliation = pd.DataFrame([
    {
        "reconciliation_id": "S6RR001",
        "reconciliation_question": "Can the historical KP 564/2022 20% maximum application-use fee be compared with SRC029 2023 reported 20% deduction prevalence?",
        "normative_source_ids": "SRC028",
        "implementation_or_policy_source_ids": "",
        "driver_evidence_source_ids": "SRC029",
        "reference_scope": "Motorcycle passenger transport; historical rule dated 4 August 2022 and later revoked.",
        "driver_evidence_period": "April-May 2023",
        "reconciliation_status": "outside_reference_scope",
        "reconciliation_conclusion": (
            "No. The historical instrument had been superseded before the SRC029 fieldwork period and cannot be used as the applicable benchmark for that observation."
        ),
        "analytical_prohibition": "Do not infer realized commission or legal compliance from the shared 20% label.",
    },
    {
        "reconciliation_id": "S6RR002",
        "reconciliation_question": "Can KP 1001/2022 be reconciled numerically with SRC029 2023 reported 20% deduction prevalence?",
        "normative_source_ids": "SRC003",
        "implementation_or_policy_source_ids": "",
        "driver_evidence_source_ids": "SRC029",
        "reference_scope": "Motorcycle passenger transport tariff regime; normative application-use fee and support-cost framework.",
        "driver_evidence_period": "April-May 2023",
        "reconciliation_status": "not_assessable",
        "reconciliation_conclusion": (
            "The available driver evidence is a reported deduction category rather than matched transaction evidence. "
            "Calculation base, covered service, and fee components are not sufficiently aligned for a realized-rate or compliance determination."
        ),
        "analytical_prohibition": "Do not treat regulatory ceilings or components as realized driver deductions.",
    },
    {
        "reconciliation_id": "S6RR003",
        "reconciliation_question": "Can KP 1001/2022 be reconciled numerically with SRC013 December 2025 reported 20% deduction prevalence?",
        "normative_source_ids": "SRC003",
        "implementation_or_policy_source_ids": "",
        "driver_evidence_source_ids": "SRC013",
        "reference_scope": "Motorcycle passenger transport tariff regime; normative application-use fee and support-cost framework.",
        "driver_evidence_period": "1-15 December 2025",
        "reconciliation_status": "not_assessable",
        "reconciliation_conclusion": (
            "The SRC013 value is driver-reported prevalence across multiple platforms rather than matched transaction evidence. "
            "The evidence cannot establish a realized deduction rate or legal-compliance result."
        ),
        "analytical_prohibition": "Do not convert reported prevalence into a transaction-level deduction rate.",
    },
    {
        "reconciliation_id": "S6RR004",
        "reconciliation_question": "Can the 2026 planned/stated 8% two-wheel passenger policy be treated as a trend comparison against SRC029 2023 or SRC013 2025 reported 20% categories?",
        "normative_source_ids": "SRC005",
        "implementation_or_policy_source_ids": "SRC026",
        "driver_evidence_source_ids": "SRC013;SRC029",
        "reference_scope": "Two-wheel passenger-service implementation discussed for 1 July 2026 onward.",
        "driver_evidence_period": "April-May 2023 and December 2025",
        "reconciliation_status": "outside_reference_scope",
        "reconciliation_conclusion": (
            "No. The driver observations predate the stated 2026 implementation and are not matched to the same service, platform, transaction base, or regulatory period."
        ),
        "analytical_prohibition": "Do not describe 20% versus 8% as a measured reduction in realized commission.",
    },
    {
        "reconciliation_id": "S6RR005",
        "reconciliation_question": "Does the Grab statement of a maximum 8% commission from 1 July 2026 establish realized driver deductions?",
        "normative_source_ids": "",
        "implementation_or_policy_source_ids": "SRC026",
        "driver_evidence_source_ids": "",
        "reference_scope": "Grab two-wheel transport; company-stated implementation.",
        "driver_evidence_period": "No matched post-implementation transaction evidence in the validated analytical evidence.",
        "reconciliation_status": "not_assessable",
        "reconciliation_conclusion": (
            "No realized deduction can be measured because the validated evidence contains no matched post-implementation driver transaction record."
        ),
        "analytical_prohibition": "Platform-stated policy must remain separate from observed_driver_deduction evidence.",
    },
    {
        "reconciliation_id": "S6RR006",
        "reconciliation_question": "Can the Gojek GoSend application-fee help page be generalized to reported motorcycle passenger deductions?",
        "normative_source_ids": "",
        "implementation_or_policy_source_ids": "SRC027",
        "driver_evidence_source_ids": "SRC013;SRC029",
        "reference_scope": "GoSend-specific fee definitions.",
        "driver_evidence_period": "2023 and 2025 multi-platform ojol evidence.",
        "reconciliation_status": "outside_reference_scope",
        "reconciliation_conclusion": (
            "No. The service-specific help-page evidence is service-specific and cannot be generalized to GoRide, GoFood, or a universal driver commission."
        ),
        "analytical_prohibition": "Do not convert customer-side or service-specific fee definitions into a universal driver deduction.",
    },
    {
        "reconciliation_id": "S6RR007",
        "reconciliation_question": "Is the broader 2026 online-transport rule sufficiently settled in the available evidence to serve as an operative universal benchmark?",
        "normative_source_ids": "SRC004;SRC006;SRC007",
        "implementation_or_policy_source_ids": "SRC005;SRC026",
        "driver_evidence_source_ids": "",
        "reference_scope": "Official announcements and status updates covering partly different services and legal layers.",
        "driver_evidence_period": "Source-registry status through 27 August 2026.",
        "reconciliation_status": "not_assessable",
        "reconciliation_conclusion": (
            "No universal operative benchmark is established by the available evidence. Official communications retain a status/scope tension, "
            "and the referenced KP-PHB 532/2026 primary legal text was not verified in the source registry."
        ),
        "analytical_prohibition": "Do not merge announcements, platform statements, and unrecovered legal text into one realized or legally operative rate.",
    },
])

evidence_gaps = pd.DataFrame([
    {
        "gap_id": "S6GAP001",
        "gap": "Complete same-observation gross-to-net chain is unavailable.",
        "affected_research_questions": "RQ1;RQ4;RQ5;RQ6;RQ7",
        "evidence_basis": "S5FND006;S5V022",
        "analytical_effect": (
            "driver_receipts_before_operating_cost, driver_side_platform_deduction, and fuel_cost are missing from a complete same-observation chain."
        ),
        "closure_treatment": "Carry as a material evidence boundary; do not impute or substitute source-defined net income.",
        "intended_report_destination": "Methodology / limitations — net reconstruction",
    },
    {
        "gap_id": "S6GAP002",
        "gap": "Matched transaction-level realized driver deduction evidence is absent.",
        "affected_research_questions": "RQ1;RQ6",
        "evidence_basis": "S5FND004;S2C008;S2C017",
        "analytical_effect": "Reported deduction categories and platform/regulatory statements cannot produce a realized deduction rate.",
        "closure_treatment": "Keep regulatory, platform-stated, driver-reported, and observed transaction layers separate.",
        "intended_report_destination": "Methodology / limitations — deductions",
    },
    {
        "gap_id": "S6GAP003",
        "gap": "Separate defensible driver fuel-cost observation is unavailable for the project net chain.",
        "affected_research_questions": "RQ1;RQ2;RQ6",
        "evidence_basis": "S5FND002;S5FND006",
        "analytical_effect": "Fuel cannot be isolated from the source-defined mixed fuel-plus-food/drink bundle for project-net subtraction.",
        "closure_treatment": "Do not model fuel from reference prices without an eligible distance-efficiency chain.",
        "intended_report_destination": "Methodology / limitations — fuel",
    },
    {
        "gap_id": "S6GAP004",
        "gap": "Available cross-source cost bundle mixes work-related fuel with personal food/drink expenditure.",
        "affected_research_questions": "RQ1;RQ2",
        "evidence_basis": "S5FND002",
        "analytical_effect": "The 31.0% and 46.0% source-defined shares cannot be labeled project operating-cost shares.",
        "closure_treatment": "Retain only as source-defined mixed spending evidence.",
        "intended_report_destination": "Results / limitations — cost boundary",
    },
    {
        "gap_id": "S6GAP005",
        "gap": "Working-hour basis remains source_reported_unspecified for the SRC029 hourly rate.",
        "affected_research_questions": "RQ4;RQ5;RQ6",
        "evidence_basis": "S5FND005;S4UE001",
        "analytical_effect": "The hourly rate cannot be relabeled as online-hour or productive-hour earnings.",
        "closure_treatment": "Retain the exact denominator label.",
        "intended_report_destination": "Methodology / results — time denominator",
    },
    {
        "gap_id": "S6GAP006",
        "gap": "Distance basis is unresolved for a defensible per-kilometer earnings metric.",
        "affected_research_questions": "RQ4;RQ5",
        "evidence_basis": "S2C012;S4-MD008",
        "analytical_effect": "Paid-trip distance cannot silently stand in for total work-related operating distance.",
        "closure_treatment": "No per-kilometer rate is derived.",
        "intended_report_destination": "Methodology / limitations — distance",
    },
    {
        "gap_id": "S6GAP007",
        "gap": "No directly_comparable cross-source evidence use exists.",
        "affected_research_questions": "RQ3;RQ7",
        "evidence_basis": "S5V020",
        "analytical_effect": "Cross-source numerical figures remain comparison-with-caveat evidence, not pooled or national estimates.",
        "closure_treatment": "Preserve comparable_with_caveat labels.",
        "intended_report_destination": "Methodology — comparability",
    },
    {
        "gap_id": "S6GAP008",
        "gap": "Twenty-four metric-specific denominators remain unresolved.",
        "affected_research_questions": "RQ3;RQ7",
        "evidence_basis": "S5V018",
        "analytical_effect": "Affected respondent-count interpretations require exclusion or explicit caveat.",
        "closure_treatment": "Carry the denominator caveat into synthesis and reporting.",
        "intended_report_destination": "Methodology / limitations — denominators",
    },
    {
        "gap_id": "S6GAP009",
        "gap": "SRC013 reports an unresolved 62-versus-67 kabupaten/kota locality count.",
        "affected_research_questions": "RQ3;RQ7",
        "evidence_basis": "S5V019",
        "analytical_effect": "No single SRC013 locality count can be asserted as certain.",
        "closure_treatment": "Retain the discrepancy explicitly.",
        "intended_report_destination": "Methodology / limitations — source consistency",
    },
    {
        "gap_id": "S6GAP010",
        "gap": "Comparable platform-, service-, and vehicle-stratified economic outcomes are insufficient.",
        "affected_research_questions": "RQ3;RQ7",
        "evidence_basis": "Analysis eligibility assessment; S5FND001-S5FND007",
        "analytical_effect": "Observed differences cannot be defensibly attributed to platform, service, or vehicle as comparative effects.",
        "closure_treatment": "Limit synthesis to source/sample evidence.",
        "intended_report_destination": "Limitations — heterogeneity",
    },
    {
        "gap_id": "S6GAP011",
        "gap": "Available 2026 regulatory evidence does not establish one universal operative online-transport benchmark.",
        "affected_research_questions": "RQ1;RQ6;RQ7",
        "evidence_basis": "SRC004;SRC005;SRC006;SRC007;SRC026",
        "analytical_effect": "Announcements and platform statements cannot be collapsed into a universal legal or realized deduction rate.",
        "closure_treatment": "Use evidence-layered reconciliation and not_assessable/outside_reference_scope outcomes.",
        "intended_report_destination": "Methodology / context — regulatory reconciliation",
    },
])

methodological_decisions = pd.DataFrame([
    {
        "decision_id": "S6-MD001",
        "decision": "Use validated analytical outputs as the synthesis basis without recomputing earlier results.",
        "rationale": "Preserving validated analytical outputs prevents inconsistent recalculation and unsupported reconstruction.",
        "evidence_basis": "S5FND001-S5FND007;S4UE001;S4UE002",
        "analytical_implication": "Synthesis references validated results and documented evidence boundaries directly.",
        "intended_report_destination": "Methodology — provenance",
    },
    {
        "decision_id": "S6-MD002",
        "decision": "Close all seven research questions with explicit evidence-bounded synthesis statuses.",
        "rationale": "Unanswered or partially answered research questions are valid analytical outcomes.",
        "evidence_basis": "Research design;S5FND001-S5FND007",
        "analytical_implication": "No unsupported answer is forced where evidence is incomplete.",
        "intended_report_destination": "Methodology / results — research-question closure",
    },
    {
        "decision_id": "S6-MD003",
        "decision": "Create no new economic metric in the analytical synthesis.",
        "rationale": "Available evidence does not support additional defensible numeric derivation beyond the validated rates and findings.",
        "evidence_basis": "Analysis eligibility assessment;S4UE001;S4UE002;S5FND001-S5FND007",
        "analytical_implication": "No CPI transform, project-net reconstruction, per-km rate, realized deduction rate, or sensitivity scenario is introduced.",
        "intended_report_destination": "Methodology — analytical boundary",
    },
    {
        "decision_id": "S6-MD004",
        "decision": "Reconcile regulatory and deduction evidence only by evidence layer, period, service, platform, and scope.",
        "rationale": "Normative rules, official implementation statements, platform policy, driver reports, and realized transactions are not interchangeable.",
        "evidence_basis": "Research-design regulatory-reconciliation framework;source registry;S2C017",
        "analytical_implication": "Reconciliation may legitimately end as not_assessable or outside_reference_scope.",
        "intended_report_destination": "Methodology / context — regulatory reconciliation",
    },
    {
        "decision_id": "S6-MD005",
        "decision": "Do not compare 2023/2025 reported 20% deduction prevalence with 2026 stated 8% policy as a measured trend.",
        "rationale": "The periods, services, platforms, evidence layers, and transaction bases are not aligned.",
        "evidence_basis": "SRC005;SRC013;SRC026;SRC029;S5FND004.",
        "analytical_implication": "No measured commission reduction is claimed.",
        "intended_report_destination": "Results / context — deductions",
    },
    {
        "decision_id": "S6-MD006",
        "decision": "Preserve project net operating earnings as not computable.",
        "rationale": "The complete same-observation gross-to-net chain remains incomplete.",
        "evidence_basis": "S5FND006;S5V022.",
        "analytical_implication": "Source-defined net income and mixed spending bundles are not substituted.",
        "intended_report_destination": "Methodology / limitations — net reconstruction",
    },
    {
        "decision_id": "S6-MD007",
        "decision": "Treat material evidence gaps as analytical outputs rather than filling them with assumptions.",
        "rationale": "Missing evidence is not zero and unsupported imputation would change the meaning of the evidence.",
        "evidence_basis": "Research-design validation gates;S5V018-S5V022",
        "analytical_implication": "The final analytical closure documents what remains unmeasured.",
        "intended_report_destination": "Limitations",
    },
    {
        "decision_id": "S6-MD008",
        "decision": "Do not force an overall welfare or economic-sustainability conclusion.",
        "rationale": "Project net is unavailable, samples are heterogeneous, and no common directly comparable national basis exists.",
        "evidence_basis": "S5FND006;S5V020;research-design synthesis gate",
        "analytical_implication": "Final reporting must distinguish bounded evidence from broader sustainability claims.",
        "intended_report_destination": "Discussion / conclusion boundary",
    },
])

A.mkdir(parents=True, exist_ok=True)
M.mkdir(parents=True, exist_ok=True)

research_question_synthesis.to_csv(A / "stage6_research_question_synthesis.csv", index=False)
regulatory_reconciliation.to_csv(M / "stage6_regulatory_reconciliation.csv", index=False)
evidence_gaps.to_csv(M / "stage6_evidence_gap_register.csv", index=False)
methodological_decisions.to_csv(M / "stage6_methodological_decision_log.csv", index=False)

print(
    f"Created synthesis outputs: {len(research_question_synthesis)} research questions | "
    f"{len(regulatory_reconciliation)} regulatory reconciliations | "
    f"{len(evidence_gaps)} evidence gaps | {len(methodological_decisions)} decisions"
)


Created synthesis outputs: 7 research questions | 7 regulatory reconciliations | 11 evidence gaps | 8 decisions


## Validation

In [4]:

def split_ids(value):
    return {item.strip() for item in str(value).split(";") if item.strip()}

validation_rows = []

def add_pass(check_id, check_name, condition, evidence="", analytical_implication=""):
    validation_rows.append({
        "check_id": check_id,
        "check_name": check_name,
        "status": "PASS" if bool(condition) else "FAIL",
        "severity": "blocking",
        "evidence": str(evidence),
        "analytical_implication": analytical_implication,
    })

def add_caveat(check_id, check_name, evidence, analytical_implication):
    validation_rows.append({
        "check_id": check_id,
        "check_name": check_name,
        "status": "CAVEAT",
        "severity": "non_blocking",
        "evidence": str(evidence),
        "analytical_implication": analytical_implication,
    })

add_pass(
    "S6V001",
    "Validated analytical finding set is complete",
    len(s5_findings) == 7,
    f"findings={len(s5_findings)}",
)
add_pass(
    "S6V002",
    "Required analytical and reference inputs are present",
    input_integrity_verified,
    f"verified_inputs={len(EXPECTED_BLOBS)}",
)
add_pass(
    "S6V003",
    "Prior analytical closure valid",
    s5_closure.iloc[0]["closure_status"] == "PASS_WITH_CAVEAT",
    s5_closure.iloc[0]["closure_status"],
)
add_pass(
    "S6V004",
    "No blocking failure in prior analytical validation",
    not ((s5_validation["status"] == "FAIL") & (s5_validation["severity"] == "blocking")).any(),
)
add_pass(
    "S6V005",
    "Expected analytical counts are preserved",
    len(s5_findings) == 7 and len(s5_registry) == 4 and len(s5_decisions) == 8 and len(s5_validation) == 22,
    "7 findings / 4 figures / 8 decisions / 22 checks",
)
add_pass(
    "S6V006",
    "All seven research questions have explicit synthesis outcomes",
    set(research_question_synthesis["research_question_id"]) == {f"RQ{i}" for i in range(1, 8)},
    f"{len(research_question_synthesis)} rows",
)
add_pass(
    "S6V007",
    "Research-question synthesis statuses are bounded",
    set(research_question_synthesis["synthesis_status"]) <= {
        "partially_supported", "not_computable", "not_assessable"
    },
    sorted(research_question_synthesis["synthesis_status"].unique()),
)

valid_evidence_ids = set(s5_findings["finding_id"]) | set(s4_unit["unit_economics_id"])
rq_evidence_ids = set()
for value in research_question_synthesis["evidence_ids"]:
    rq_evidence_ids |= split_ids(value)

add_pass(
    "S6V008",
    "Research-question evidence IDs resolve",
    rq_evidence_ids <= valid_evidence_ids,
    sorted(rq_evidence_ids - valid_evidence_ids),
)

valid_gap_ids = set(evidence_gaps["gap_id"])
rq_gap_ids = set()
for value in research_question_synthesis["remaining_gap_ids"]:
    rq_gap_ids |= split_ids(value)

add_pass(
    "S6V009",
    "Research-question gap IDs resolve",
    rq_gap_ids <= valid_gap_ids,
    sorted(rq_gap_ids - valid_gap_ids),
)

stage1_source_ids = set(s1_sources["source_id"])
rq_source_ids = set()
for value in research_question_synthesis["source_ids"]:
    rq_source_ids |= split_ids(value)

add_pass(
    "S6V010",
    "Research-question source IDs resolve",
    rq_source_ids <= stage1_source_ids,
    sorted(rq_source_ids - stage1_source_ids),
)

ue = s4_unit.set_index("unit_economics_id")
unit_values_ok = (
    abs(float(ue.loc["S4UE001", "derived_value"]) - 15272.73) < 0.01
    and abs(float(ue.loc["S4UE002", "derived_value"]) - 16800.00) < 0.01
    and set(s4_unit["derivation_basis"]) == {"ratio_of_source_means"}
)
add_pass(
    "S6V011",
    "Validated gross unit-economics values are preserved",
    unit_values_ok,
    "S4UE001=15272.73; S4UE002=16800.00; ratio_of_source_means",
)

add_pass(
    "S6V012",
    "No project-net answer is manufactured",
    research_question_synthesis.loc[
        research_question_synthesis["research_question_id"] == "RQ4", "synthesis_status"
    ].iloc[0] == "not_computable",
    "RQ4=not_computable",
)

add_pass(
    "S6V013",
    "No per-kilometer earnings metric introduced",
    not research_question_synthesis["synthesis_answer"].str.contains(
        r"Rp[\d,.]+\s*per kilometer", regex=True
    ).any(),
)

add_pass(
    "S6V014",
    "Deferred CPI comparisons remain deferred",
    set(
        s3_eligibility.loc[
            s3_eligibility["primary_analysis_eligibility"]
            == "not_eligible_for_primary_transformed_comparison",
            "comparison_id",
        ]
    ) == {"S2C001", "S2C002", "S2C005"},
    "S2C001/S2C002/S2C005",
)

allowed_reconciliation_statuses = {
    "consistent_with_reference",
    "apparent_difference_requires_reconciliation",
    "not_assessable",
    "outside_reference_scope",
}
add_pass(
    "S6V015",
    "Regulatory reconciliation statuses are valid",
    set(regulatory_reconciliation["reconciliation_status"]) <= allowed_reconciliation_statuses,
    sorted(regulatory_reconciliation["reconciliation_status"].unique()),
)

regulatory_source_ids = set()
for col in [
    "normative_source_ids",
    "implementation_or_policy_source_ids",
    "driver_evidence_source_ids",
]:
    for value in regulatory_reconciliation[col]:
        regulatory_source_ids |= split_ids(value)

add_pass(
    "S6V016",
    "Regulatory reconciliation source IDs resolve",
    regulatory_source_ids <= stage1_source_ids,
    sorted(regulatory_source_ids - stage1_source_ids),
)

add_pass(
    "S6V017",
    "No matched realized-evidence consistency claim is made",
    not regulatory_reconciliation["reconciliation_status"].isin(
        ["consistent_with_reference", "apparent_difference_requires_reconciliation"]
    ).any(),
    "All rows end as not_assessable or outside_reference_scope.",
)

rr004 = regulatory_reconciliation.set_index("reconciliation_id").loc["S6RR004"]
add_pass(
    "S6V018",
    "2026 stated 8% policy is not treated as a measured trend against earlier driver reports",
    rr004["reconciliation_status"] == "outside_reference_scope"
    and "Do not describe 20% versus 8%" in rr004["analytical_prohibition"],
    rr004["reconciliation_status"],
)

add_pass(
    "S6V019",
    "Evidence-gap register is complete for the synthesis design",
    set(evidence_gaps["gap_id"]) == {f"S6GAP{i:03d}" for i in range(1, 12)},
    f"{len(evidence_gaps)} gaps",
)

add_pass(
    "S6V020",
    "Eight methodological decisions are traceable",
    set(methodological_decisions["decision_id"]) == {f"S6-MD{i:03d}" for i in range(1, 9)},
    f"{len(methodological_decisions)} decisions",
)

add_pass(
    "S6V021",
    "No new numerical economic result table is created",
    not any(
        col in research_question_synthesis.columns
        for col in ["numeric_value", "derived_value", "modelled_value", "scenario_value"]
    ),
    "Outputs are synthesis, reconciliation, evidence-gap, and validation records.",
)

add_caveat(
    "S6V022",
    "Unresolved denominators carried forward",
    "24",
    "Affected respondent-count interpretations remain caveated.",
)
add_caveat(
    "S6V023",
    "SRC013 locality discrepancy carried forward",
    "62/67 kabupaten/kota",
    "Do not assert one locality count as certain.",
)
add_caveat(
    "S6V024",
    "No directly comparable cross-source evidence",
    "0 directly_comparable uses",
    "Cross-source numerical figures remain comparable_with_caveat.",
)
add_caveat(
    "S6V025",
    "SRC010 recall/current-period distinction remains material",
    "retrospective_recall + contemporaneous",
    "Do not interpret recalled periods as independent historical waves.",
)
add_caveat(
    "S6V026",
    "Project net operating earnings remain unavailable",
    "driver_receipts_before_operating_cost; driver_side_platform_deduction; fuel_cost",
    "No project-net reconstruction or source-net substitution.",
)
add_caveat(
    "S6V027",
    "SRC029 hourly rate retains source-reported working-hour basis",
    "source_reported_unspecified",
    "Do not relabel as online or productive hours.",
)
add_caveat(
    "S6V028",
    "Universal 2026 regulatory benchmark remains unresolved in the available evidence",
    "SRC004/SRC005/SRC006/SRC007/SRC026",
    "Keep legal text, implementation statements, platform policy, and realized driver evidence separate.",
)

stage6_validation = pd.DataFrame(validation_rows)

blocking_failures = stage6_validation[
    (stage6_validation["status"] == "FAIL")
    & (stage6_validation["severity"] == "blocking")
]
assert blocking_failures.empty, blocking_failures.to_dict("records")

stage6_validation.to_csv(M / "stage6_synthesis_validation.csv", index=False)

counts = stage6_validation["status"].value_counts().to_dict()
print(
    "Synthesis validation: "
    f"PASS={counts.get('PASS', 0)} | CAVEAT={counts.get('CAVEAT', 0)} | FAIL={counts.get('FAIL', 0)}"
)


Synthesis validation: PASS=21 | CAVEAT=7 | FAIL=0


## Analytical Closure

In [5]:

stage6_manifest = pd.DataFrame([
    {
        "output_id": "S6-OUT001",
        "output_path": "data/analytical/stage6_research_question_synthesis.csv",
        "output_type": "research_question_synthesis",
        "row_count": str(len(research_question_synthesis)),
        "role": "Evidence-bounded synthesis of the seven research questions.",
    },
    {
        "output_id": "S6-OUT002",
        "output_path": "metadata/stage6_regulatory_reconciliation.csv",
        "output_type": "regulatory_reconciliation",
        "row_count": str(len(regulatory_reconciliation)),
        "role": "Layered rule, implementation, platform-policy, and driver-evidence reconciliation.",
    },
    {
        "output_id": "S6-OUT003",
        "output_path": "metadata/stage6_evidence_gap_register.csv",
        "output_type": "evidence_gap_register",
        "row_count": str(len(evidence_gaps)),
        "role": "Material evidence gaps that constrain final analytical claims.",
    },
    {
        "output_id": "S6-OUT004",
        "output_path": "metadata/stage6_methodological_decision_log.csv",
        "output_type": "methodological_decision_log",
        "row_count": str(len(methodological_decisions)),
        "role": "Material synthesis and closure decisions.",
    },
    {
        "output_id": "S6-OUT005",
        "output_path": "metadata/stage6_synthesis_validation.csv",
        "output_type": "synthesis_validation",
        "row_count": str(len(stage6_validation)),
        "role": "Analytical, traceability, and boundary validation.",
    },
])

stage6_manifest.to_csv(M / "stage6_output_manifest.csv", index=False)

manifest_ok = True
manifest_errors = []

for row in stage6_manifest.itertuples(index=False):
    path = REPO_DIR / row.output_path
    if not path.is_file():
        manifest_ok = False
        manifest_errors.append(f"missing:{row.output_path}")
        continue
    actual_rows = len(pd.read_csv(path, dtype=str, keep_default_na=False))
    if actual_rows != int(row.row_count):
        manifest_ok = False
        manifest_errors.append(
            f"row_count:{row.output_path}:{actual_rows}!={row.row_count}"
        )

closure_rows = []

def add_closure(check_id, check_name, condition, evidence, analytical_implication):
    closure_rows.append({
        "check_id": check_id,
        "check_name": check_name,
        "status": "PASS" if bool(condition) else "FAIL",
        "severity": "blocking",
        "evidence": str(evidence),
        "analytical_implication": analytical_implication,
    })

add_closure(
    "S6CL001",
    "No blocking synthesis-validation failure",
    blocking_failures.empty,
    f"blocking_failures={len(blocking_failures)}",
    "Final analytical closure requires zero blocking failures.",
)
add_closure(
    "S6CL002",
    "Manifest outputs exist with expected row counts",
    manifest_ok,
    ";".join(manifest_errors) if manifest_errors else "all manifest outputs verified",
    "Closure requires reproducible synthesis outputs.",
)
add_closure(
    "S6CL003",
    "All seven research questions have explicit closure outcomes",
    len(research_question_synthesis) == 7
    and research_question_synthesis["synthesis_status"].ne("").all(),
    f"research_questions={len(research_question_synthesis)}",
    "No research question is silently omitted.",
)
add_closure(
    "S6CL004",
    "Regulatory reconciliation preserves evidence layers and scope",
    set(regulatory_reconciliation["reconciliation_status"])
    <= {"not_assessable", "outside_reference_scope"},
    sorted(regulatory_reconciliation["reconciliation_status"].unique()),
    "No legal-compliance or realized-rate conclusion is manufactured.",
)
add_closure(
    "S6CL005",
    "Project net operating earnings remain not computable",
    research_question_synthesis.loc[
        research_question_synthesis["research_question_id"] == "RQ4",
        "synthesis_status",
    ].iloc[0] == "not_computable",
    "RQ4=not_computable",
    "No source-defined net substitution or missing-component imputation.",
)
add_closure(
    "S6CL006",
    "Material evidence gaps are explicitly preserved",
    len(evidence_gaps) == 11,
    f"evidence_gaps={len(evidence_gaps)}",
    "Analytical limitations remain traceable into final reporting.",
)
add_closure(
    "S6CL007",
    "No prohibited new economic metric is introduced",
    stage6_validation.set_index("check_id").loc["S6V021", "status"] == "PASS",
    "S6V021=PASS",
    "The analysis remains synthesis and closure rather than unsupported numerical expansion.",
)

stage6_closure_validation = pd.DataFrame(closure_rows)
assert (stage6_closure_validation["status"] == "PASS").all(), (
    stage6_closure_validation[stage6_closure_validation["status"] != "PASS"]
    .to_dict("records")
)

stage6_closure_validation.to_csv(M / "stage6_closure_validation.csv", index=False)

validation_counts = stage6_validation["status"].value_counts().to_dict()
closure_status = (
    "PASS_WITH_CAVEAT"
    if validation_counts.get("CAVEAT", 0) > 0
    else "PASS"
)

stage6_closure_summary = pd.DataFrame([{
    "analysis_component": "Analytical Synthesis & Evidence-Gap Closure",
    "closure_status": closure_status,
    "research_question_count": len(research_question_synthesis),
    "regulatory_reconciliation_count": len(regulatory_reconciliation),
    "evidence_gap_count": len(evidence_gaps),
    "methodological_decision_count": len(methodological_decisions),
    "validation_check_count": len(stage6_validation),
    "validation_pass_count": validation_counts.get("PASS", 0),
    "validation_caveat_count": validation_counts.get("CAVEAT", 0),
    "validation_fail_count": validation_counts.get("FAIL", 0),
    "blocking_failure_count": len(blocking_failures),
    "closure_check_count": len(stage6_closure_validation),
    "closure_pass_count": int((stage6_closure_validation["status"] == "PASS").sum()),
    "key_conclusion": (
        "The validated evidence supports bounded source-specific descriptive findings and two SRC029 gross unit-economics rates, "
        "but does not support project net operating earnings, realized transaction deduction rates, per-kilometer net earnings, "
        "a sensitivity model, or an overall economic-sustainability conclusion."
    ),
}])

stage6_closure_summary.to_csv(M / "stage6_closure_summary.csv", index=False)

print(
    f"Analytical closure: {closure_status} | "
    f"research_questions={len(research_question_synthesis)} | "
    f"regulatory_reconciliations={len(regulatory_reconciliation)} | "
    f"evidence_gaps={len(evidence_gaps)} | "
    f"validation PASS={validation_counts.get('PASS', 0)} "
    f"CAVEAT={validation_counts.get('CAVEAT', 0)} "
    f"FAIL={validation_counts.get('FAIL', 0)} | "
    f"closure={len(stage6_closure_validation)}/{len(stage6_closure_validation)} PASS"
)


Analytical closure: PASS_WITH_CAVEAT | research_questions=7 | regulatory_reconciliations=7 | evidence_gaps=11 | validation PASS=21 CAVEAT=7 FAIL=0 | closure=7/7 PASS
